In [1]:
import numpy as np
from numba import cuda

@cuda.jit
def bitonic_step(arr, j, k):
    i = cuda.grid(1)

    if i < arr.size:
        ixj = i ^ j

        if ixj > i:
            if ((i & k) == 0 and arr[i] > arr[ixj]) or \
               ((i & k) != 0 and arr[i] < arr[ixj]):
                arr[i], arr[ixj] = arr[ixj], arr[i]

# Input (size must be power of 2)
arr = np.array([10, 30, 11, 20, 4, 330, 21, 110], dtype=np.int32)

d_arr = cuda.to_device(arr)

threads = 256
blocks = (len(arr) + threads - 1) // threads

k = 2
while k <= len(arr):
    j = k // 2
    while j > 0:
        bitonic_step[blocks, threads](d_arr, j, k)
        cuda.synchronize()
        j //= 2
    k *= 2

print("Sorted:", d_arr.copy_to_host())

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Sorted: [  4  10  11  20  21  30 110 330]
